# Task 4: The Turing Test - Part 1: The Super-Imposter
## Genetic Algorithm for Adversarial Text Evolution

**Objective:** Use a Genetic Algorithm (GA) to evolve AI-generated text until it can fool our best detector (DistilBERT-LoRA achieving 99.71% accuracy)

**Prerequisites:** Complete **Task 2.3 (Tier C)** and **Task 3** first!

---

## The Challenge:

Our DistilBERT-LoRA model achieves:
- **99.71% accuracy**
- **100% AI recall** (catches all AI text)
- **0.40% FPR** (only 2 human false positives)

Can we use evolutionary algorithms to create a "Super-Imposter" paragraph that fools this detector?

---

## Genetic Algorithm Approach:

1. **Initial Population:** 10 AI-generated Victorian-style paragraphs
2. **Fitness Function:** Human probability from DistilBERT (higher = better)
3. **Selection:** Keep top 3 paragraphs each generation
4. **Mutation:** LLM-based rewriting with Victorian authenticity prompts
5. **Evolution:** 10 generations
6. **Goal:** Achieve >85% "Human" confidence score

---

## What We Learned from XAI (Task 3):

Our detector identifies:
- **Victorian markers:** 'ere', past tense, complex syntax
- **Modern AI tells:** 'however' overuse, article density, present tense
- **Distributed patterns:** Not single words but structural flow
- **Function words:** AI uses more 'the/a/of', less 'I/he/was/had'

We'll use these insights to guide our mutation strategies!

---

## Expected Outcomes:

**Success (>85% Human):** Detector vulnerable to adversarial evolution  
**Failure (<85% Human):** Detector robust against evolutionary attacks  
**Both results are valuable research findings!**

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install transformers peft torch pandas numpy scikit-learn matplotlib seaborn google-generativeai -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


## 2. Import Libraries

In [2]:
import google.generativeai as genai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import random
import os
import time
from typing import List, Dict, Tuple
from datetime import datetime

# Transformers imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🔧 Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

/tmp/ipykernel_336830/1087084035.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ All libraries imported successfully!
🔧 PyTorch version: 2.8.0+cu128
🔧 Device: CPU


## 3. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Model paths
MODEL_DIR = "/home/avani/precog/reports/lora_distilbert/lora_adapter"  # Update if needed!
BASE_MODEL = "distilbert-base-uncased"
MAX_LENGTH = 512

# Gemini API configuration
# IMPORTANT: Get your API key from https://makersuite.google.com/app/apikey
GEMINI_API_KEY = "YOUR_API_KEY_HERE"  # ⚠️ REPLACE THIS!

# Genetic Algorithm parameters
POPULATION_SIZE = 10
NUM_GENERATIONS = 10
TOP_K_SELECTION = 3
TARGET_FITNESS = 0.85  # >85% "Human" confidence
MUTATION_RATE = 1.0  # Probability of mutation (100% for all offspring)

# Output configuration
OUTPUT_DIR = "task4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 80)
print("TASK 4 CONFIGURATION")
print("=" * 80)
print(f"Model directory: {MODEL_DIR}")
print(f"Base model: {BASE_MODEL}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Device: {device}")
print(f"\nGenetic Algorithm Parameters:")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations: {NUM_GENERATIONS}")
print(f"  Selection: Top {TOP_K_SELECTION}")
print(f"  Target fitness: >{TARGET_FITNESS:.0%} Human confidence")
print("=" * 80)

# Verify model exists
if os.path.exists(MODEL_DIR):
    print(f"\n✅ Model found at: {MODEL_DIR}")
else:
    print(f"\n❌ WARNING: Model not found at: {MODEL_DIR}")
    print(f"   Update MODEL_DIR or retrain the model in Task 2.3 (Tier C)")

## 4. Configure Gemini API

**⚠️ IMPORTANT:** You need a Gemini API key to run this task.

Get your free API key here: https://makersuite.google.com/app/apikey

Then update `GEMINI_API_KEY` in the configuration cell above.

In [ ]:
# Check if API key is set
if GEMINI_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️" * 40)
    print("\n❌ ERROR: Gemini API key not configured!")
    print("\n📝 To fix this:")
    print("   1. Go to https://makersuite.google.com/app/apikey")
    print("   2. Create a new API key (free)")
    print("   3. Update GEMINI_API_KEY in the configuration cell above")
    print("   4. Re-run this cell")
    print("\n⚠️" * 40)
    raise ValueError("Gemini API key not configured")

# Configure Gemini
try:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-pro')
    
    # Test the API
    test_response = gemini_model.generate_content("Say 'API working!'")
    
    print("✅ Gemini API configured successfully!")
    print(f"🤖 Model: gemini-pro")
    print(f"🧪 Test response: {test_response.text}")
    
except Exception as e:
    print(f"❌ Error configuring Gemini API: {e}")
    print("\n💡 Common issues:")
    print("   - Invalid API key (check for typos)")
    print("   - API key not activated yet (wait a few minutes)")
    print("   - Rate limit exceeded (wait and try again)")
    raise

## 5. Load Fine-Tuned DistilBERT-LoRA Model

In [ ]:
print("=" * 80)
print("LOADING FINE-TUNED DISTILBERT-LORA MODEL")
print("=" * 80)

# Load tokenizer
print(f"\n📂 Loading tokenizer from: {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(f"   ✓ Tokenizer loaded")

# Load base model
print(f"\n📂 Loading base model: {BASE_MODEL}")
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1}
)
print(f"   ✓ Base model loaded")

# Load LoRA adapter
print(f"\n📂 Loading LoRA adapter from: {MODEL_DIR}")
distilbert_model = PeftModel.from_pretrained(base_model, MODEL_DIR)
print(f"   ✓ LoRA adapter loaded")

# Move to device and set to evaluation mode
distilbert_model = distilbert_model.to(device)
distilbert_model.eval()

print(f"\n✅ DistilBERT-LoRA model ready!")
print(f"   Device: {device}")
print(f"   Mode: Evaluation")
print(f"   Expected accuracy: 99.71%")
print(f"   Expected AI recall: 100%")

## 6. Create Prediction Wrapper for DistilBERT

This wrapper makes our model compatible with sklearn-style `predict_proba()` interface.

In [ ]:
class DistilBERTPredictor:
    """
    Wrapper for DistilBERT-LoRA model with sklearn-style predict_proba interface.
    """
    
    def __init__(self, model, tokenizer, device, max_length=512):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length
    
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        """
        Predict class probabilities for texts.
        
        Args:
            texts: List of text strings
        
        Returns:
            Array of shape (n_samples, 2) with [prob_human, prob_ai]
        """
        # Tokenize
        encodings = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors='pt'
        )
        
        # Move to device
        encodings = {k: v.to(self.device) for k, v in encodings.items()}
        
        # Get predictions
        with torch.no_grad():
            outputs = self.model(**encodings)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        
        return probs  # Returns [prob_human, prob_ai] for each text
    
    def predict(self, texts: List[str]) -> np.ndarray:
        """
        Predict class labels for texts.
        
        Returns:
            Array of predicted labels (0=Human, 1=AI)
        """
        probs = self.predict_proba(texts)
        return np.argmax(probs, axis=1)

# Create predictor instance
distilbert_predictor = DistilBERTPredictor(
    model=distilbert_model,
    tokenizer=tokenizer,
    device=device,
    max_length=MAX_LENGTH
)

print("✅ DistilBERT predictor created!")

# Test it
test_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "It was a dark and stormy night when Holmes deduced the truth."
]

test_probs = distilbert_predictor.predict_proba(test_texts)

print(f"\n🧪 Test predictions:")
for text, probs in zip(test_texts, test_probs):
    prob_human = probs[0]
    prob_ai = probs[1]
    predicted = "Human" if prob_human > prob_ai else "AI"
    print(f"\n   Text: {text[:60]}...")
    print(f"   Human: {prob_human:.4f} ({prob_human*100:.2f}%)")
    print(f"   AI:    {prob_ai:.4f} ({prob_ai*100:.2f}%)")
    print(f"   Predicted: {predicted}")

## 7. Define Victorian Topics for Initial Population

These topics match our human corpus (Arthur Conan Doyle & Robert Louis Stevenson).

In [ ]:
# Victorian topics from our corpus
TOPICS = [
    # Arthur Conan Doyle themes
    "The Science of Deduction",
    "The Curse of Ancestry",
    "Vengeance from the Past",
    "Domestic Tragedy and Hidden Crimes",
    "The Deception of Appearances",
    
    # Robert Louis Stevenson themes
    "The Duality of Human Nature",
    "The Violation of the Dead",
    "Science vs. The Unexplained",
    "The Terror of the Sea",
    "Folklore and Superstition"
]

print("📚 Victorian Topics for Initial Population:")
print("=" * 80)
for i, topic in enumerate(TOPICS, 1):
    print(f"  {i:2d}. {topic}")
print("=" * 80)

## 8. Generate Initial Population

Create 10 diverse AI-generated paragraphs using Victorian themes and authentic stylistic markers.

In [ ]:
# Authors to emulate
AUTHORS = ['Arthur Conan Doyle', 'Robert Louis Stevenson']

# Generate initial prompts
INITIAL_PROMPTS = []

for i in range(POPULATION_SIZE):
    topic = TOPICS[i]
    author = AUTHORS[i % 2]  # Alternate between authors
    
    prompt = f"""Write a 100-150 word paragraph about '{topic}' in the style of {author}.

CRITICAL INSTRUCTIONS FOR VICTORIAN AUTHENTICITY:
- Use archaic Victorian vocabulary: 'ere' (before), 'lest' (for fear that), 'thence' (from there), 'wherefore' (why), 'whence' (from where)
- Write EXCLUSIVELY in PAST TENSE: use 'was', 'had', 'were', 'had been', 'had seen'
- AVOID modern transitions: 'however', 'therefore', 'additionally', 'furthermore', 'moreover'
- Use complex Victorian sentence structures with semicolons (;) and em-dashes (—)
- Include first-person pronouns: 'I', 'he', 'she', 'we' (not impersonal 'it is' style)
- Write as if published in 1890s London for a literary magazine
- Use fewer articles ('the', 'a') than modern prose
- Include sensory details and atmospheric descriptions

Topic: {topic}

Output only the paragraph text (no title, no introduction):""".
    
    INITIAL_PROMPTS.append(prompt)

print(f"✅ Created {len(INITIAL_PROMPTS)} initial prompts")
print(f"\n📝 Example prompt:")
print("=" * 80)
print(INITIAL_PROMPTS[0])
print("=" * 80)

In [ ]:
print("=" * 80)
print("GENERATING INITIAL POPULATION")
print("=" * 80)
print(f"Creating {POPULATION_SIZE} AI-generated Victorian paragraphs...\n")

initial_population = []

for i, prompt in enumerate(INITIAL_PROMPTS, 1):
    try:
        print(f"🤖 Generating paragraph {i}/{POPULATION_SIZE}...")
        response = gemini_model.generate_content(prompt)
        text = response.text.strip()
        
        # Check word count
        word_count = len(text.split())
        print(f"   ✓ Generated ({word_count} words)")
        
        initial_population.append(text)
        
        # Rate limiting
        time.sleep(1)
        
    except Exception as e:
        print(f"   ✗ Error: {e}")
        # If generation fails, create a fallback
        fallback_text = f"""It was a dark and stormy night when I first encountered the mystery of {TOPICS[i-1]}. 
        The events which followed were to prove most singular, and I confess that they troubled my mind 
        for many months thereafter. Had I but known what lay in store, I should never have ventured forth 
        that fateful evening. Yet such is the nature of fate that we cannot perceive its workings ere 
        the moment has passed."""
        initial_population.append(fallback_text)
        time.sleep(2)

print(f"\n{'='*80}")
print(f"✅ Initial population created: {len(initial_population)} individuals")
print(f"{'='*80}")

# Show first example
print(f"\n📝 Example from initial population:")
print(f"{'='*80}")
print(initial_population[0])
print(f"{'='*80}")

## 9. Define Fitness Function

**Fitness = Probability of being classified as "Human"**

Higher fitness = more human-like = better for fooling the detector.

In [ ]:
def calculate_fitness(text: str, model) -> float:
    """
    Calculate how 'human-like' the text appears to DistilBERT.
    
    Args:
        text: Input text to evaluate
        model: DistilBERT predictor with predict_proba method
    
    Returns:
        float: Probability of being classified as Human (0.0 to 1.0)
               Higher = more human-like = better fitness
    """
    try:
        probs = model.predict_proba([text])[0]
        prob_human = probs[0]  # Probability of Human class
        return prob_human
    except Exception as e:
        print(f"Error in fitness calculation: {e}")
        return 0.0


def evaluate_population(population: List[str], model) -> List[Dict]:
    """
    Evaluate fitness for entire population.
    
    Args:
        population: List of text strings
        model: DistilBERT predictor
    
    Returns:
        List of dicts with keys: 'id', 'text', 'fitness', 'prob_human', 'prob_ai', 'predicted_class'
        Sorted by fitness (descending)
    """
    results = []
    
    for i, text in enumerate(population):
        probs = model.predict_proba([text])[0]
        prob_human = probs[0]
        prob_ai = probs[1]
        
        results.append({
            'id': i,
            'text': text,
            'fitness': prob_human,
            'prob_human': prob_human,
            'prob_ai': prob_ai,
            'predicted_class': 'Human' if prob_human > prob_ai else 'AI'
        })
    
    # Sort by fitness (descending)
    return sorted(results, key=lambda x: x['fitness'], reverse=True)


print("✅ Fitness functions defined!")
print("\n📊 Fitness = P(Human | text)")
print("   Higher fitness → More human-like → Better for fooling detector")

## 10. Define Selection Strategy

Select the top K individuals (highest fitness) to survive and reproduce.

In [ ]:
def select_top_k(evaluated_pop: List[Dict], k: int = 3) -> List[str]:
    """
    Select top K individuals based on fitness.
    
    Args:
        evaluated_pop: List of evaluated individuals (sorted by fitness)
        k: Number of top individuals to select
    
    Returns:
        List of text strings from top performers
    """
    top_k = evaluated_pop[:k]
    return [ind['text'] for ind in top_k]


print(f"✅ Selection strategy: Top-{TOP_K_SELECTION} (elitism)")
print(f"   Only the fittest {TOP_K_SELECTION} individuals survive each generation")

## 11. Define Mutation Strategies

LLM-based mutations informed by our XAI analysis (Task 3).

In [ ]:
# Mutation prompts informed by XAI findings
MUTATION_PROMPTS = [
    # Strategy 0: Victorian Authenticity Markers (HIGHEST PRIORITY)
    """CRITICAL: Enhance this paragraph with AUTHENTIC Victorian markers:

REQUIRED CHANGES:
1. Add archaic conjunctions: 'ere' (before), 'lest' (for fear that), 'thence' (from there)
2. Convert ALL verbs to past tense: 'was', 'had', 'were', 'had been', 'had seen'
3. REMOVE modern transitions: Replace 'however' with 'yet' or 'nevertheless'
4. Add first-person pronouns: 'I observed', 'he remarked', 'she whispered'
5. Reduce articles: Use fewer 'the' and 'a' than modern AI would
6. Use semicolons to connect related thoughts (Victorian style)

Original paragraph:
{text}

Victorian-enhanced paragraph (100-150 words):""",
    
    # Strategy 1: Rhythm change
    """Rewrite the following paragraph to change the rhythm and flow of sentences 
    while preserving the vocabulary and meaning. Make some sentences shorter, others longer. 
    Vary the sentence structure. Keep Victorian style and archaic words like 'ere', 'lest'.
    
    Original paragraph:
    {text}
    
    Rewritten paragraph (100-150 words):""",
    
    # Strategy 2: Add archaic elements
    """Take this paragraph and enhance it with MORE Victorian archaic vocabulary and 
    constructions. Add words like 'ere', 'lest', 'thence', 'whence', 'wherefore'. 
    Use past perfect tense ('had been', 'had seen'). Make it sound like authentic 1890s prose.
    
    Original paragraph:
    {text}
    
    Enhanced Victorian paragraph (100-150 words):""",
    
    # Strategy 3: Grammatical variation
    """Rewrite this paragraph with subtle grammatical variations - use inverted sentence 
    structures, insert em-dashes for dramatic pauses, add semicolons for complex thoughts. 
    Introduce one minor grammatical inconsistency that a human Victorian author might make. 
    Keep the Victorian style.
    
    Original paragraph:
    {text}
    
    Grammatically varied paragraph (100-150 words):""",
    
    # Strategy 4: Detective fiction style
    """Rewrite this paragraph to sound MORE like Arthur Conan Doyle's Sherlock Holmes stories. 
    Use deductive reasoning language, observational details, Victorian London atmosphere. 
    Include past tense verbs, complex sentence structures, archaic conjunctions. 
    Avoid modern words like 'however', 'therefore'.
    
    Original paragraph:
    {text}
    
    Holmes-style paragraph (100-150 words):""",
    
    # Strategy 5: Stylistic variation
    """Add natural stylistic variation to this paragraph as a Victorian author would. 
    Include some dialogue if appropriate, vary punctuation (use semicolons, em-dashes), 
    add sensory details, use metaphor or simile. Keep Victorian vocabulary and past tense.
    
    Original paragraph:
    {text}
    
    Stylistically varied paragraph (100-150 words):""",
    
    # Strategy 6: Temporal consistency
    """Rewrite this paragraph to have perfect temporal consistency - ensure all verbs 
    are in past tense, maintain consistent point of view, create flow between sentences 
    as if part of a longer Victorian narrative. Use 'was', 'had', 'were' frequently.
    
    Original paragraph:
    {text}
    
    Temporally consistent paragraph (100-150 words):""",
]


def mutate_text(text: str, gemini_model, mutation_type: str = 'random') -> str:
    """
    Mutate text using Gemini LLM with specified strategy.
    
    Args:
        text: Original paragraph
        gemini_model: Gemini generative model
        mutation_type: 'random' or specific strategy index
    
    Returns:
        Mutated paragraph text
    """
    if mutation_type == 'random':
        prompt_template = np.random.choice(MUTATION_PROMPTS)
    else:
        prompt_template = MUTATION_PROMPTS[0]  # Default to Victorian enhancement
    
    prompt = prompt_template.format(text=text)
    
    try:
        response = gemini_model.generate_content(prompt)
        mutated_text = response.text.strip()
        
        # Ensure reasonable length (100-150 words)
        word_count = len(mutated_text.split())
        
        if word_count < 80 or word_count > 200:
            # If too short/long, try again
            print(f"      ⚠️  Mutation produced {word_count} words, using original...")
            return text
        
        return mutated_text
    
    except Exception as e:
        print(f"      ✗ Error in mutation: {e}")
        return text  # Return original if mutation fails


print("✅ Mutation strategies defined!")
print(f"\n🧬 {len(MUTATION_PROMPTS)} mutation strategies available:")
print("   0. Victorian Authenticity Enhancement (XAI-informed)")
print("   1. Rhythm and Flow Variation")
print("   2. Archaic Vocabulary Addition")
print("   3. Grammatical Variation")
print("   4. Detective Fiction Style")
print("   5. Stylistic Variation")
print("   6. Temporal Consistency")

## 12. Genetic Algorithm Main Loop

The core evolutionary algorithm that iteratively improves text fitness.

In [ ]:
def run_genetic_algorithm(
    initial_population: List[str],
    distilbert_model,
    gemini_model,
    num_generations: int = 10,
    top_k: int = 3,
    target_fitness: float = 0.85
) -> Dict:
    """
    Run the genetic algorithm for adversarial text generation.
    
    Args:
        initial_population: List of initial text samples
        distilbert_model: DistilBERT predictor
        gemini_model: Gemini LLM for mutations
        num_generations: Number of generations to evolve
        top_k: Number of top individuals to select each generation
        target_fitness: Target fitness to achieve (>85% Human)
    
    Returns:
        Dict with keys: 'history', 'best_individual', 'success', 'generations'
    """
    
    population = initial_population.copy()
    history = []
    
    print("=" * 80)
    print("GENETIC ALGORITHM: EVOLVING AI TEXT TO FOOL DISTILBERT")
    print("=" * 80)
    print(f"Initial population: {len(population)} individuals")
    print(f"Generations: {num_generations}")
    print(f"Selection: Top {top_k} survivors per generation")
    print(f"Target fitness: >{target_fitness:.0%} Human confidence")
    print("=" * 80)
    
    for generation in range(1, num_generations + 1):
        print(f"\n{'=' * 80}")
        print(f"GENERATION {generation}")
        print(f"{'=' * 80}")
        
        # Evaluate fitness
        evaluated = evaluate_population(population, distilbert_model)
        
        # Track statistics
        best = evaluated[0]
        worst = evaluated[-1]
        avg_fitness = np.mean([ind['fitness'] for ind in evaluated])
        
        # Log generation statistics
        gen_stats = {
            'generation': generation,
            'best_fitness': best['fitness'],
            'worst_fitness': worst['fitness'],
            'avg_fitness': avg_fitness,
            'best_text': best['text'],
            'best_predicted_class': best['predicted_class'],
            'population_size': len(population)
        }
        history.append(gen_stats)
        
        # Print statistics
        print(f"\n📊 GENERATION {generation} STATISTICS:")
        print(f"   Best fitness:    {best['fitness']:.4f} ({best['fitness']*100:.2f}% Human)")
        print(f"   Average fitness: {avg_fitness:.4f} ({avg_fitness*100:.2f}% Human)")
        print(f"   Worst fitness:   {worst['fitness']:.4f} ({worst['fitness']*100:.2f}% Human)")
        print(f"   Best classified as: {best['predicted_class']}")
        
        # Check if target achieved
        if best['fitness'] >= target_fitness:
            print(f"\n{'🎉' * 40}")
            print(f"🎉 SUCCESS! Achieved >{target_fitness:.0%} Human confidence!")
            print(f"{'🎉' * 40}")
            print(f"\n   Final fitness: {best['fitness']:.4f} ({best['fitness']*100:.2f}%)")
            print(f"\n📝 WINNING PARAGRAPH:")
            print(f"{'=' * 80}")
            print(f"{best['text']}")
            print(f"{'=' * 80}")
            
            return {
                'success': True,
                'generations': generation,
                'best_individual': best,
                'history': history
            }
        
        # Selection: Keep top K
        survivors = select_top_k(evaluated, k=top_k)
        
        print(f"\n🔍 TOP {top_k} SURVIVORS:")
        for i, ind in enumerate(evaluated[:top_k], 1):
            print(f"   #{i}: {ind['fitness']:.4f} ({ind['fitness']*100:.2f}% Human)")
            print(f"        Text: {ind['text'][:100]}...")
        
        # Generate next population through mutation
        print(f"\n🧬 MUTATING TOP {top_k} SURVIVORS...")
        next_population = survivors.copy()  # Keep elites
        
        # Create mutations of each survivor
        num_offspring = (len(initial_population) - top_k) // top_k
        
        for i, survivor_text in enumerate(survivors):
            for j in range(num_offspring):
                print(f"   Mutating survivor #{i+1}, offspring {j+1}/{num_offspring}...")
                mutated = mutate_text(survivor_text, gemini_model, mutation_type='random')
                next_population.append(mutated)
                time.sleep(1)  # Rate limiting
        
        # Ensure population size
        while len(next_population) < len(initial_population):
            parent = np.random.choice(survivors)
            mutated = mutate_text(parent, gemini_model)
            next_population.append(mutated)
            time.sleep(1)
        
        population = next_population[:len(initial_population)]  # Cap at initial size
        print(f"   ✓ New population size: {len(population)}")
    
    # Final generation completed without reaching target
    print(f"\n{'=' * 80}")
    print(f"GENETIC ALGORITHM COMPLETED ({num_generations} GENERATIONS)")
    print(f"{'=' * 80}")
    
    final_evaluated = evaluate_population(population, distilbert_model)
    final_best = final_evaluated[0]
    
    print(f"\n📊 FINAL RESULTS:")
    print(f"   Best fitness achieved: {final_best['fitness']:.4f} ({final_best['fitness']*100:.2f}% Human)")
    print(f"   Target fitness: {target_fitness:.4f} ({target_fitness*100:.2f}% Human)")
    print(f"   Success: {'YES ✅' if final_best['fitness'] >= target_fitness else 'NO ❌'}")
    
    return {
        'success': final_best['fitness'] >= target_fitness,
        'generations': num_generations,
        'best_individual': final_best,
        'history': history
    }


print("✅ Genetic algorithm function defined!")

## 13. Run the Genetic Algorithm

**This cell will take ~15-20 minutes to run** (10 generations with API calls).

⏰ Expected time: 1-2 minutes per generation

In [ ]:
# Record start time
start_time = datetime.now()
print(f"🕐 Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n⏰ This will take approximately {NUM_GENERATIONS * 2} minutes...\n")

# Run the GA
results = run_genetic_algorithm(
    initial_population=initial_population,
    distilbert_model=distilbert_predictor,
    gemini_model=gemini_model,
    num_generations=NUM_GENERATIONS,
    top_k=TOP_K_SELECTION,
    target_fitness=TARGET_FITNESS
)

# Record end time
end_time = datetime.now()
elapsed = end_time - start_time

print(f"\n{'=' * 80}")
print(f"✅ GENETIC ALGORITHM COMPLETED")
print(f"{'=' * 80}")
print(f"🕐 Start time:   {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🕐 End time:     {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️  Elapsed time: {elapsed}")
print(f"{'=' * 80}")

## 14. Analyze Victorian Markers

Compare linguistic features between initial and evolved text.

In [ ]:
def analyze_victorian_markers(text: str) -> Dict:
    """
    Check presence of Victorian authenticity markers.
    
    Returns:
        Dict with counts of various Victorian linguistic features
    """
    text_lower = text.lower()
    words = text.split()
    
    markers = {
        'archaic_conj': sum(text_lower.count(w) for w in ['ere', 'lest', 'thence', 'whence', 'wherefore']),
        'past_tense': sum(text_lower.count(w) for w in [' was ', ' were ', ' had ']),
        'modern_trans': sum(text_lower.count(w) for w in ['however', 'therefore', 'additionally']),
        'first_person': sum(text_lower.count(w) for w in [' i ', ' he ', ' she ']),
        'article_count': text.count(' the ') + text.count(' a '),
        'article_density': (text.count(' the ') + text.count(' a ')) / len(words) if words else 0,
        'semicolons': text.count(';'),
        'em_dashes': text.count('—') + text.count('--'),
        'word_count': len(words)
    }
    
    return markers


print("✅ Victorian marker analysis function defined!")

In [ ]:
# Get initial best vs evolved best
initial_evaluated = evaluate_population(initial_population, distilbert_predictor)
initial_best = initial_evaluated[0]
final_best = results['best_individual']

# Analyze markers
initial_markers = analyze_victorian_markers(initial_best['text'])
final_markers = analyze_victorian_markers(final_best['text'])

print("\n" + "=" * 80)
print("VICTORIAN MARKER ANALYSIS: Initial Best vs. Evolved Best")
print("=" * 80)

print(f"\n📊 FITNESS IMPROVEMENT:")
print(f"   Initial: {initial_best['fitness']:.4f} ({initial_best['fitness']*100:.2f}% Human)")
print(f"   Evolved: {final_best['fitness']:.4f} ({final_best['fitness']*100:.2f}% Human)")
print(f"   Gain:    {final_best['fitness'] - initial_best['fitness']:+.4f} ({(final_best['fitness'] - initial_best['fitness'])*100:+.2f}%)")

print(f"\n📈 LINGUISTIC FEATURE CHANGES:")
print(f"{'Marker':<25s} {'Initial':>10s} {'Evolved':>10s} {'Change':>10s}")
print(f"{'-'*25} {'-'*10} {'-'*10} {'-'*10}")

for marker in initial_markers.keys():
    initial_val = initial_markers[marker]
    final_val = final_markers[marker]
    change = final_val - initial_val
    
    if marker == 'article_density':
        print(f"{marker:<25s} {initial_val:>10.4f} {final_val:>10.4f} {change:>+10.4f}")
    else:
        print(f"{marker:<25s} {initial_val:>10.0f} {final_val:>10.0f} {change:>+10.0f}")

print("\n" + "=" * 80)

## 15. Visualize Evolution

In [ ]:
def visualize_evolution(history: List[Dict], target: float = 0.85):
    """
    Create visualization of GA evolution over generations.
    """
    generations = [h['generation'] for h in history]
    best_fitness = [h['best_fitness'] for h in history]
    avg_fitness = [h['avg_fitness'] for h in history]
    worst_fitness = [h['worst_fitness'] for h in history]
    
    plt.figure(figsize=(14, 8))
    
    # Plot fitness evolution
    plt.plot(generations, best_fitness, 'g-o', label='Best', linewidth=2.5, markersize=8)
    plt.plot(generations, avg_fitness, 'b--s', label='Average', linewidth=2, markersize=6)
    plt.plot(generations, worst_fitness, 'r:^', label='Worst', linewidth=1.5, markersize=5)
    
    # Target line
    plt.axhline(y=target, color='purple', linestyle='--', linewidth=2, label=f'Target ({target:.0%})')
    
    # Decision boundary
    plt.axhline(y=0.50, color='gray', linestyle=':', linewidth=1.5, label='Decision Boundary (50%)')
    
    # Fill between best and worst
    plt.fill_between(generations, worst_fitness, best_fitness, alpha=0.2, color='blue')
    
    plt.xlabel('Generation', fontsize=14, fontweight='bold')
    plt.ylabel('Fitness (Human Probability)', fontsize=14, fontweight='bold')
    plt.title('Genetic Algorithm Evolution: Adversarial Text Generation\nFooling DistilBERT-LoRA (99.71% Accuracy)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1.05)
    plt.xlim(0.5, max(generations) + 0.5)
    
    # Add annotations
    max_fitness = max(best_fitness)
    max_gen = generations[best_fitness.index(max_fitness)]
    plt.annotate(f'Best: {max_fitness:.2%}\nGen {max_gen}', 
                 xy=(max_gen, max_fitness), 
                 xytext=(max_gen + 0.5, max_fitness - 0.1),
                 arrowprops=dict(arrowstyle='->', color='green', lw=2),
                 fontsize=11, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))
    
    plt.tight_layout()
    
    # Save figure
    output_path = os.path.join(OUTPUT_DIR, 'task4_ga_evolution.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {output_path}")
    
    plt.show()


# Create visualization
visualize_evolution(results['history'], target=TARGET_FITNESS)

## 16. Analyze Successful Mutations

In [ ]:
def analyze_successful_mutations(history: List[Dict]):
    """
    Analyze which generations had the most successful mutations.
    """
    improvements = []
    
    for i in range(1, len(history)):
        improvement = history[i]['best_fitness'] - history[i-1]['best_fitness']
        improvements.append({
            'generation': history[i]['generation'],
            'improvement': improvement,
            'fitness': history[i]['best_fitness'],
            'from_fitness': history[i-1]['best_fitness']
        })
    
    # Sort by improvement
    top_improvements = sorted(improvements, key=lambda x: x['improvement'], reverse=True)[:5]
    
    print("\n" + "=" * 80)
    print("📈 TOP 5 MOST SUCCESSFUL MUTATIONS")
    print("=" * 80)
    
    for i, imp in enumerate(top_improvements, 1):
        print(f"\n#{i}: Generation {imp['generation']}")
        print(f"   Improvement: {imp['improvement']:+.4f} ({imp['improvement']*100:+.2f}%)")
        print(f"   From: {imp['from_fitness']:.4f} → To: {imp['fitness']:.4f}")
    
    print("\n" + "=" * 80)
    
    # Visualize improvements
    plt.figure(figsize=(12, 6))
    
    gens = [imp['generation'] for imp in improvements]
    imps = [imp['improvement'] for imp in improvements]
    
    colors = ['green' if x > 0 else 'red' for x in imps]
    
    plt.bar(gens, imps, color=colors, alpha=0.7, edgecolor='black')
    plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
    plt.xlabel('Generation', fontsize=12, fontweight='bold')
    plt.ylabel('Fitness Improvement', fontsize=12, fontweight='bold')
    plt.title('Generation-to-Generation Fitness Improvements', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    output_path = os.path.join(OUTPUT_DIR, 'task4_improvements.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {output_path}")
    
    plt.show()


# Analyze mutations
analyze_successful_mutations(results['history'])

## 17. Save Results

In [ ]:
# Save generation history as CSV
results_df = pd.DataFrame(results['history'])
csv_path = os.path.join(OUTPUT_DIR, 'task4_ga_results.csv')
results_df.to_csv(csv_path, index=False)
print(f"✅ Saved generation history: {csv_path}")

# Save best evolved paragraph
best_text_path = os.path.join(OUTPUT_DIR, 'task4_best_evolved_paragraph.txt')
with open(best_text_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("BEST EVOLVED PARAGRAPH (SUPER-IMPOSTER)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Fitness: {results['best_individual']['fitness']:.4f} ({results['best_individual']['fitness']*100:.2f}% Human)\n")
    f.write(f"Predicted Class: {results['best_individual']['predicted_class']}\n")
    f.write(f"Generations: {results['generations']}\n")
    f.write(f"Success: {'YES' if results['success'] else 'NO'}\n")
    f.write("\n" + "=" * 80 + "\n\n")
    f.write(results['best_individual']['text'])
    f.write("\n\n" + "=" * 80 + "\n")

print(f"✅ Saved best paragraph: {best_text_path}")

# Save initial population for comparison
initial_path = os.path.join(OUTPUT_DIR, 'task4_initial_population.txt')
with open(initial_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("INITIAL POPULATION (10 AI-Generated Paragraphs)\n")
    f.write("=" * 80 + "\n\n")
    for i, text in enumerate(initial_population, 1):
        f.write(f"\n{'-' * 80}\n")
        f.write(f"PARAGRAPH {i}\n")
        f.write(f"{'-' * 80}\n")
        f.write(text)
        f.write("\n")

print(f"✅ Saved initial population: {initial_path}")

print(f"\n📁 All results saved to: {OUTPUT_DIR}/")

## 18. Final Report and Interpretation

In [ ]:
print("\n" + "=" * 80)
print("TASK 4: FINAL REPORT - THE SUPER-IMPOSTER GENETIC ALGORITHM")
print("=" * 80)

print(f"\n📊 EXPERIMENT SUMMARY:")
print(f"   Target detector: DistilBERT-LoRA (99.71% accuracy)")
print(f"   Initial population: {POPULATION_SIZE} AI-generated paragraphs")
print(f"   Generations evolved: {results['generations']}")
print(f"   Selection pressure: Top-{TOP_K_SELECTION} survivors")
print(f"   Mutation strategy: LLM-based (Gemini Pro)")
print(f"   Target fitness: >{TARGET_FITNESS:.0%} Human confidence")

print(f"\n📈 RESULTS:")
initial_evaluated = evaluate_population(initial_population, distilbert_predictor)
initial_best = initial_evaluated[0]
final_best = results['best_individual']

print(f"   Initial best fitness: {initial_best['fitness']:.4f} ({initial_best['fitness']*100:.2f}% Human)")
print(f"   Final best fitness:   {final_best['fitness']:.4f} ({final_best['fitness']*100:.2f}% Human)")
print(f"   Improvement:          {final_best['fitness'] - initial_best['fitness']:+.4f} ({(final_best['fitness'] - initial_best['fitness'])*100:+.2f}%)")
print(f"   Target achieved:      {'✅ YES' if results['success'] else '❌ NO'}")

print(f"\n{'='*80}")

if results['success']:
    print("\n🎉 SUCCESS: GENETIC ALGORITHM FOOLED THE DETECTOR!")
    print("=" * 80)
    print(f"\n✅ Achieved {final_best['fitness']:.2%} Human confidence in {results['generations']} generations")
    print(f"\n📝 WINNING 'SUPER-IMPOSTER' PARAGRAPH:")
    print("=" * 80)
    print(final_best['text'])
    print("=" * 80)
    
    print(f"\n🔬 INTERPRETATION:")
    print(f"   ➤ DistilBERT-LoRA is vulnerable to adversarial evolution")
    print(f"   ➤ Iterative LLM-based refinement can bypass the detector")
    print(f"   ➤ Victorian style patterns can be learned/mimicked by modern AI")
    print(f"   ➤ Detector needs adversarial training for robustness")
    
    print(f"\n💡 IMPLICATIONS:")
    print(f"   • No AI detector is perfect - adversarial attacks are possible")
    print(f"   • Genetic algorithms can systematically explore the detector's weaknesses")
    print(f"   • Production systems need continuous updating with adversarial examples")
    print(f"   • Red-teaming (adversarial testing) is essential for deployment")

else:
    print("\n🛡️  DETECTOR ROBUST: GENETIC ALGORITHM FAILED TO REACH TARGET")
    print("=" * 80)
    print(f"\n❌ Best fitness: {final_best['fitness']:.2%} (Target: {TARGET_FITNESS:.0%})")
    print(f"\n📝 BEST EVOLVED PARAGRAPH (Still detected as AI):")
    print("=" * 80)
    print(final_best['text'])
    print("=" * 80)
    
    print(f"\n🔬 INTERPRETATION:")
    print(f"   ➤ DistilBERT-LoRA is robust against evolutionary attacks")
    print(f"   ➤ Deep structural patterns cannot be mimicked by surface-level mutations")
    print(f"   ➤ Victorian authenticity markers (ere, past tense, function words) hard to fake")
    print(f"   ➤ Distributed detection strategy is adversarially robust")
    
    print(f"\n💡 IMPLICATIONS:")
    print(f"   • The detector learned deep structural features, not superficial patterns")
    print(f"   • Even with 10 generations of evolution, AI cannot replicate Victorian prose")
    print(f"   • This demonstrates the value of domain-specific training (Victorian literature)")
    print(f"   • The model may be suitable for production deployment")

print(f"\n{'='*80}")
print(f"\n🎓 KEY TAKEAWAY:")
print(f"   Both success and failure are valuable research findings!")
print(f"   Success → Shows attack vectors to defend against")
print(f"   Failure → Demonstrates detector robustness")
print(f"\n{'='*80}")

## 19. Side-by-Side Comparison

In [ ]:
print("\n" + "=" * 80)
print("SIDE-BY-SIDE COMPARISON: Initial vs. Evolved")
print("=" * 80)

initial_evaluated = evaluate_population(initial_population, distilbert_predictor)
initial_best = initial_evaluated[0]
final_best = results['best_individual']

print(f"\n{'INITIAL BEST':<40s} | {'EVOLVED BEST'}")
print(f"{'-'*40} | {'-'*40}")
print(f"Fitness: {initial_best['fitness']:.4f} ({initial_best['fitness']*100:.2f}%) | Fitness: {final_best['fitness']:.4f} ({final_best['fitness']*100:.2f}%)")
print(f"Predicted: {initial_best['predicted_class']:<26s} | Predicted: {final_best['predicted_class']}")
print(f"{'='*40} | {'='*40}")

print(f"\n📝 INITIAL BEST PARAGRAPH:")
print("=" * 80)
print(initial_best['text'])
print("=" * 80)

print(f"\n📝 EVOLVED BEST PARAGRAPH:")
print("=" * 80)
print(final_best['text'])
print("=" * 80)

# Analyze differences
initial_markers = analyze_victorian_markers(initial_best['text'])
final_markers = analyze_victorian_markers(final_best['text'])

print(f"\n🔍 WHAT CHANGED?")
print(f"{'Marker':<25s} {'Initial':>10s} {'Evolved':>10s} {'Change':>10s}")
print(f"{'-'*25} {'-'*10} {'-'*10} {'-'*10}")

key_markers = ['archaic_conj', 'past_tense', 'modern_trans', 'first_person', 'article_density', 'semicolons']
for marker in key_markers:
    initial_val = initial_markers[marker]
    final_val = final_markers[marker]
    change = final_val - initial_val
    
    if marker == 'article_density':
        print(f"{marker:<25s} {initial_val:>10.4f} {final_val:>10.4f} {change:>+10.4f}")
    else:
        print(f"{marker:<25s} {initial_val:>10.0f} {final_val:>10.0f} {change:>+10.0f}")

print("\n" + "=" * 80)

## 20. Conclusion and Next Steps

---

## ✅ Task 4 Complete!

You have successfully:
1. ✅ Implemented a Genetic Algorithm for adversarial text generation
2. ✅ Used Gemini LLM for Victorian-style mutations
3. ✅ Evolved AI text over 10 generations
4. ✅ Tested robustness of DistilBERT-LoRA detector
5. ✅ Analyzed linguistic features of evolved text
6. ✅ Created comprehensive visualizations

---

## 🎓 What We Learned:

**About Adversarial ML:**
- Genetic algorithms can systematically probe detector weaknesses
- LLM-based mutations are more effective than random perturbations
- Evolution pressure drives toward human-like patterns

**About Our Detector:**
- Robustness can be quantified through adversarial testing
- Domain-specific training (Victorian literature) may increase robustness
- Deep structural patterns are harder to mimic than surface features

**About Victorian Style:**
- Archaic vocabulary ('ere', 'lest') is measurable and trackable
- Past tense consistency is a strong Victorian marker
- Function word patterns are harder to mimic than content words

---

## 📁 Output Files:

All results saved in `task4_outputs/`:
- `task4_ga_evolution.png` - Fitness evolution over generations
- `task4_improvements.png` - Generation-to-generation improvements
- `task4_ga_results.csv` - Complete generation history
- `task4_best_evolved_paragraph.txt` - Final Super-Imposter text
- `task4_initial_population.txt` - Starting population for comparison

---

## 🚀 Future Work:

1. **Multi-objective optimization:** Optimize for both fitness and readability
2. **Larger populations:** Test with 50-100 individuals
3. **More generations:** Run for 20-50 generations
4. **Crossover operators:** Combine features from multiple parents
5. **Adversarial training:** Retrain detector on evolved examples
6. **Human evaluation:** Ask humans to judge evolved text
7. **Different LLMs:** Test with GPT-4, Claude, or Llama for mutations

---

## 📚 Research Impact:

This work demonstrates:
- **Red-teaming methodology** for AI detectors
- **Adversarial robustness testing** with evolutionary algorithms
- **Domain-specific stylometry** (Victorian literature)
- **LLM-based mutation operators** for text evolution

Both success and failure are publishable results!

---

**Great work! You've completed the adversarial testing phase of your AI detection project! 🎉**